# Join & Analise de Capacidade Produtiva 2021-2026
### Bases: VolumeDeAtendimentos | TempoMedioAtendimentos | ColaboradoresPorDia | Previsao Prophet 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

## 1. Carregamento das Bases

In [ ]:
def normalizar_empresa(serie):
    return serie.str.upper().str.replace(r'EMP0*(\d+)', r'EMP\1', regex=True)

df_vol   = pd.read_csv('../Bases/VolumeDeAtendimentos.csv',   sep=';', parse_dates=['data'], index_col=0)
df_tempo = pd.read_csv('../Bases/TempoMedioAtendimentos.csv', sep=';', parse_dates=['data'])
df_colab = pd.read_csv('../Bases/ColaboradoresPorDia.csv',    sep=';', parse_dates=['data'])
df_prev  = pd.read_csv('../Bases/previsao_volume_2026.csv',          parse_dates=['data'])

df_vol['empresa']   = normalizar_empresa(df_vol['empresa'])
df_tempo['empresa'] = normalizar_empresa(df_tempo['empresa'])
df_colab['empresa'] = normalizar_empresa(df_colab['empresa'])

print(f'VolumeDeAtendimentos  : {df_vol.shape}')
print(f'TempoMedioAtendimentos: {df_tempo.shape}')
print(f'ColaboradoresPorDia   : {df_colab.shape}')
print(f'Previsao 2026         : {df_prev.shape}')

## 2. Base Historica Consolidada (2021-2025)

In [ ]:
df_hist = (
    df_vol
    .merge(df_tempo, on=['data', 'empresa', 'produto'], how='left')
    .merge(df_colab, on=['data', 'empresa'],            how='left')
)
df_hist['tipo'] = 'Historico'

print(f'Linhas: {len(df_hist):,}')
print(f'Periodo: {df_hist["data"].min().date()} -> {df_hist["data"].max().date()}')
print(f'Nulos tempo_medio_min        : {df_hist["tempo_medio_min"].isna().sum()}')
print(f'Nulos quantidade_colaboradores: {df_hist["quantidade_colaboradores"].isna().sum()}')
df_hist.head()

## 3. Projecao 2026 com Prophet

A previsao Prophet retorna o **volume total diario** (soma de todas as empresas e produtos).  
Para desagregar no nivel empresa x produto, aplicamos as **proporcoes historicas de 2025** sobre cada dia previsto.

In [ ]:
# Proporcao de cada (empresa, produto) sobre o total de 2025
proporcoes = (
    df_vol[df_vol['data'].dt.year == 2025]
    .groupby(['empresa', 'produto'])['volume_atendimentos']
    .sum()
)
proporcoes = (proporcoes / proporcoes.sum()).reset_index()
proporcoes.columns = ['empresa', 'produto', 'proporcao']

# Volume Prophet diario para 2026
df_prophet_2026 = (
    df_prev[['data', 'Prophet']]
    .query("'2026-01-01' <= data <= '2026-12-31'")
    .reset_index(drop=True)
)

# Cross-join: 365 dias x 400 combinacoes (80 empresas x 5 produtos)
df_2026 = df_prophet_2026.merge(proporcoes, how='cross')
df_2026['volume_atendimentos'] = (df_2026['Prophet'] * df_2026['proporcao']).round().astype(int)
df_2026 = df_2026.drop(columns=['Prophet', 'proporcao'])

# Tempo medio 2026 = media de 2025 por (empresa, produto)
tempo_ref = (
    df_tempo[df_tempo['data'].dt.year == 2025]
    .groupby(['empresa', 'produto'])['tempo_medio_min']
    .mean().reset_index()
)
df_2026 = df_2026.merge(tempo_ref, on=['empresa', 'produto'], how='left')

# Colaboradores 2026 = media arredondada de 2025 por empresa
colab_ref = (
    df_colab[df_colab['data'].dt.year == 2025]
    .groupby('empresa')['quantidade_colaboradores']
    .mean().round().astype(int).reset_index()
)
df_2026 = df_2026.merge(colab_ref, on='empresa', how='left')
df_2026['tipo'] = 'Previsao Prophet'

print(f'Base 2026: {df_2026.shape}')
df_2026.head()

## 4. Base Unificada 2021-2026

In [ ]:
COLUNAS = ['data', 'empresa', 'produto', 'volume_atendimentos',
           'tempo_medio_min', 'quantidade_colaboradores', 'tipo']

df_unificado = (
    pd.concat([df_hist[COLUNAS], df_2026[COLUNAS]], ignore_index=True)
    .sort_values(['data', 'empresa', 'produto'])
    .reset_index(drop=True)
)

print(f'Base unificada: {len(df_unificado):,} linhas')
print(f'Periodo: {df_unificado["data"].min().date()} -> {df_unificado["data"].max().date()}')
print('\nRegistros por tipo:')
print(df_unificado['tipo'].value_counts())

df_unificado.to_csv('../Bases/base_unificada_2021_2026.csv', index=False, sep=';')
print('\nExportado: Bases/base_unificada_2021_2026.csv')
df_unificado.head(10)

## 5. Analise de Capacidade Produtiva

**Premissa:** cada colaborador tem **480 minutos disponiveis por dia** (8h uteis).  
- **Demanda (min):** soma(volume_atendimentos x tempo_medio_min) por empresa/dia  
- **Capacidade (min):** quantidade_colaboradores x 480  
- **Utilizacao (%):** demanda / capacidade x 100  
- **Gap (min):** capacidade - demanda (positivo = folga, negativo = sobrecarga)

In [ ]:
MINUTOS_POR_COLABORADOR = 480

df_cap = (
    df_unificado
    .assign(minutos_demanda=lambda d: d['volume_atendimentos'] * d['tempo_medio_min'])
    .groupby(['data', 'empresa', 'tipo'], as_index=False)
    .agg(
        volume_total  =('volume_atendimentos', 'sum'),
        demanda_min   =('minutos_demanda',     'sum'),
        colaboradores =('quantidade_colaboradores', 'first')
    )
)

df_cap['capacidade_min'] = df_cap['colaboradores'] * MINUTOS_POR_COLABORADOR
df_cap['utilizacao_pct'] = (df_cap['demanda_min'] / df_cap['capacidade_min'] * 100).round(1)
df_cap['gap_min']        = (df_cap['capacidade_min'] - df_cap['demanda_min']).round(1)
df_cap['status'] = pd.cut(
    df_cap['utilizacao_pct'],
    bins=[-np.inf, 70, 90, np.inf],
    labels=['Adequado', 'Alerta', 'Critico']
).astype(str)

print('Resumo geral por tipo:')
print(df_cap.groupby('tipo')[['utilizacao_pct', 'gap_min']].describe().round(1))
df_cap.head()

In [ ]:
util_diaria = df_cap.groupby('data')['utilizacao_pct'].mean()

fig, ax = plt.subplots(figsize=(14, 5))
util_diaria[:'2025-12-31'].plot(ax=ax, color='steelblue',  linewidth=0.8, label='Historico (2021-2025)')
util_diaria['2026-01-01':].plot(ax=ax, color='darkorange', linewidth=1.2, linestyle='--', label='Previsao Prophet 2026')
ax.axhline(90, color='tomato',    linewidth=0.9, linestyle=':', label='Limite critico (90%)')
ax.axhline(70, color='goldenrod', linewidth=0.9, linestyle=':', label='Limite de alerta (70%)')
ax.set_title('Taxa de Utilizacao Media da Capacidade Produtiva - 2021-2026')
ax.set_ylabel('Utilizacao Media (%)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b/%Y'))
plt.tight_layout()
plt.show()

In [ ]:
df_cap['ano'] = df_cap['data'].dt.year
status_ano = (
    df_cap.groupby(['ano', 'status'])
    .size()
    .unstack(fill_value=0)
    [['Adequado', 'Alerta', 'Critico']]
)
status_ano.plot(
    kind='bar', stacked=True, figsize=(10, 5),
    color={'Adequado': 'seagreen', 'Alerta': 'goldenrod', 'Critico': 'tomato'},
    title='Distribuicao de Status de Capacidade por Ano (empresa x dia)'
)
plt.ylabel('Ocorrencias (empresa x dia)')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
top_util = (
    df_cap[df_cap['ano'] == 2026]
    .groupby('empresa')['utilizacao_pct']
    .mean().sort_values(ascending=False).head(10)
)
top_util.plot(kind='bar', color='tomato',
              title='Top 10 Empresas - Maior Utilizacao Media (Previsao 2026)')
plt.ylabel('Utilizacao Media (%)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

top_gap = (
    df_cap[df_cap['ano'] == 2026]
    .groupby('empresa')['gap_min']
    .mean().sort_values().head(10)
)
top_gap.plot(kind='barh', color='tomato',
             title='Top 10 Empresas - Maior Sobrecarga Media Diaria (Previsao 2026)')
plt.xlabel('Gap Medio (min/dia) - negativo = sobrecarga')
plt.tight_layout()
plt.show()

In [ ]:
resumo_mensal = (
    df_cap[df_cap['ano'] == 2026]
    .assign(mes=lambda d: d['data'].dt.to_period('M').astype(str))
    .groupby('mes')[['utilizacao_pct', 'gap_min', 'volume_total']]
    .mean().round(1)
)
print('Resumo mensal - Previsao 2026 (media entre empresas):')
print(resumo_mensal.to_string())

## 6. Graficos Interativos — Previsao 2026

Graficos com filtro por empresa (dropdown). Valores representam a **media diaria** do mes.
- **Grafico 1:** Demanda vs Capacidade produtiva (minutos/dia)
- **Grafico 2:** Taxa de Utilizacao da Capacidade (%) com limites de alerta/critico
- **Grafico 3:** Colaboradores atuais vs Colaboradores sugeridos (`ceil(demanda / 480 min)`)

In [ ]:

import plotly.graph_objects as go

df_2026_cap = (
    df_cap[df_cap['ano'] == 2026]
    .assign(mes=lambda d: d['data'].dt.to_period('M').dt.to_timestamp())
    .groupby(['mes', 'empresa'], as_index=False)
    .agg(
        demanda_min    =('demanda_min',    'mean'),
        capacidade_min =('capacidade_min', 'mean'),
        utilizacao_pct =('utilizacao_pct', 'mean'),
        colaboradores  =('colaboradores',  'mean'),
    )
    .assign(
        demanda_min    =lambda d: d['demanda_min'].round(1),
        capacidade_min =lambda d: d['capacidade_min'].round(1),
        utilizacao_pct =lambda d: d['utilizacao_pct'].round(1),
        colaboradores  =lambda d: d['colaboradores'].round().astype(int),
    )
)
df_2026_cap['colaboradores_sugeridos'] = (
    np.ceil(df_2026_cap['demanda_min'] / MINUTOS_POR_COLABORADOR).astype(int)
)

empresas_2026 = sorted(df_2026_cap['empresa'].unique())
print(f'{len(empresas_2026)} empresas | {df_2026_cap["mes"].nunique()} meses')
df_2026_cap.head()


In [ ]:

_emp0 = empresas_2026[0]
fig1 = go.Figure()

for emp in empresas_2026:
    d = df_2026_cap[df_2026_cap['empresa'] == emp].sort_values('mes')
    vis = emp == _emp0
    fig1.add_trace(go.Scatter(
        x=d['mes'], y=d['demanda_min'],
        name='Demanda (min)', visible=vis,
        line=dict(color='steelblue', width=2), marker=dict(size=7),
        hovertemplate='%{x|%b/%Y}<br>Demanda: %{y:.0f} min<extra></extra>'
    ))
    fig1.add_trace(go.Scatter(
        x=d['mes'], y=d['capacidade_min'],
        name='Capacidade (min)', visible=vis,
        line=dict(color='seagreen', width=2, dash='dash'), marker=dict(size=7),
        hovertemplate='%{x|%b/%Y}<br>Capacidade: %{y:.0f} min<extra></extra>'
    ))

n = len(empresas_2026)
fig1.update_layout(
    title=f'Demanda vs Capacidade Produtiva (media diaria) — {_emp0} — 2026',
    yaxis_title='Minutos (media diaria)', xaxis_title='Mes',
    updatemenus=[dict(
        buttons=[
            dict(label=emp, method='update',
                 args=[{'visible': [j // 2 == i for j in range(2 * n)]},
                       {'title': f'Demanda vs Capacidade Produtiva (media diaria) — {emp} — 2026'}])
            for i, emp in enumerate(empresas_2026)
        ],
        direction='down', showactive=True,
        x=0.0, xanchor='left', y=1.22, yanchor='top',
    )],
    legend=dict(orientation='h', y=1.1, x=1, xanchor='right'),
    hovermode='x unified', height=500,
)
fig1.show()


In [ ]:

_emp0 = empresas_2026[0]
fig2 = go.Figure()

for emp in empresas_2026:
    d = df_2026_cap[df_2026_cap['empresa'] == emp].sort_values('mes')
    vis = emp == _emp0
    fig2.add_trace(go.Scatter(
        x=d['mes'], y=d['utilizacao_pct'],
        name=emp, visible=vis,
        line=dict(color='darkorange', width=2), marker=dict(size=7),
        hovertemplate='%{x|%b/%Y}<br>Utilizacao: %{y:.1f}%<extra></extra>'
    ))

n = len(empresas_2026)
fig2.update_layout(
    title=f'Taxa de Utilizacao da Capacidade — {_emp0} — 2026',
    yaxis_title='Utilizacao (%)', xaxis_title='Mes',
    updatemenus=[dict(
        buttons=[
            dict(label=emp, method='update',
                 args=[{'visible': [j == i for j in range(n)]},
                       {'title': f'Taxa de Utilizacao da Capacidade — {emp} — 2026'}])
            for i, emp in enumerate(empresas_2026)
        ],
        direction='down', showactive=True,
        x=0.0, xanchor='left', y=1.22, yanchor='top',
    )],
    shapes=[
        dict(type='line', x0=0, x1=1, xref='paper', y0=90, y1=90,
             line=dict(color='tomato', dash='dot', width=1.5)),
        dict(type='line', x0=0, x1=1, xref='paper', y0=70, y1=70,
             line=dict(color='goldenrod', dash='dot', width=1.5)),
    ],
    annotations=[
        dict(x=1, xref='paper', y=90, text='Critico (90%)',
             showarrow=False, xanchor='right', font=dict(color='tomato', size=10)),
        dict(x=1, xref='paper', y=70, text='Alerta (70%)',
             showarrow=False, xanchor='right', font=dict(color='goldenrod', size=10)),
    ],
    hovermode='x unified', height=500,
)
fig2.show()


In [ ]:

_emp0 = empresas_2026[0]
fig3 = go.Figure()

for emp in empresas_2026:
    d = df_2026_cap[df_2026_cap['empresa'] == emp].sort_values('mes')
    vis = emp == _emp0
    fig3.add_trace(go.Scatter(
        x=d['mes'], y=d['colaboradores'],
        name='Colaboradores Atuais', visible=vis,
        line=dict(color='steelblue', width=2), marker=dict(size=7),
        hovertemplate='%{x|%b/%Y}<br>Atuais: %{y}<extra></extra>'
    ))
    fig3.add_trace(go.Scatter(
        x=d['mes'], y=d['colaboradores_sugeridos'],
        name='Colaboradores Sugeridos', visible=vis,
        line=dict(color='tomato', width=2, dash='dash'), marker=dict(size=7),
        hovertemplate='%{x|%b/%Y}<br>Sugeridos: %{y}<extra></extra>'
    ))

n = len(empresas_2026)
fig3.update_layout(
    title=f'Colaboradores Atuais vs Sugeridos — {_emp0} — 2026',
    yaxis_title='Quantidade de Colaboradores', xaxis_title='Mes',
    updatemenus=[dict(
        buttons=[
            dict(label=emp, method='update',
                 args=[{'visible': [j // 2 == i for j in range(2 * n)]},
                       {'title': f'Colaboradores Atuais vs Sugeridos — {emp} — 2026'}])
            for i, emp in enumerate(empresas_2026)
        ],
        direction='down', showactive=True,
        x=0.0, xanchor='left', y=1.22, yanchor='top',
    )],
    legend=dict(orientation='h', y=1.1, x=1, xanchor='right'),
    hovermode='x unified', height=500,
)
fig3.show()
